# Repeat Purchase Prediction, part 2
# Data Preprocessing

## Notebook purpose

This notebook turns the raw tables into data a model can use. It follows `01_eda.ipynb`, which found what is wrong with the data and what can be used at prediction time. Here we decide what to do about each finding, test the decisions that are not obvious, and save the result.

The target is `repeat_purchase_90d`: whether a customer buys again within 90 days of a purchase. Since the evaluation metric that will be used is F1, some choices in this notebook like the validation process and the threshold are made with F1 in mind and not accuracy.

This notebook covers:

- Merging the customer table and fixing the problems found in the EDA (placeholders (values randomly added to fill in missing information), wrong signs, calculation inconsistencies, empty rows);
- Building a validation scheme that respects time order;
- Comparing the main treatment options (missing values, outliers, transformations, scaling, encoding) with the same models and the same folds;
- Feature engineering and feature selection;
- Saving the cleaned data, a cleaning log and the preprocessing steps in a form that can be reused on the test set.

## Analysis principles

A few rules apply to the whole notebook and most of them come from the EDA (section 13).

- **Two kinds of steps:** 
    - *Stateless* steps work row by row and need nothing learned from the data: fixing a placeholder, recomputing a mean from its own total and count, building a ratio.
    - *Stateful* steps learn something from the data (imputation values, encoding maps, scaling, feature selection) and are fitted only on the training part of each fold.
- **Time order.** The predictors describe the customer's history up to the purchase, and the target describes the next 90 days. Validation therefore uses later periods than training, with a 90-day gap between them. A random split would overstate the scores.
- **No information from later rows.** A feature may not use any later purchase of the same customer, in train or in test.
- **Decisions are tested, not assumed.** When two treatments are reasonable, they are compared with the same model and the same folds. The simpler one is kept unless the other is clearly better. Clearly better means the 95% bootstrap interval of the F1 difference, with customers (not rows) as the resampling unit, stays above zero.
- **Everything that changes data is logged.** Each repair records the column, the action, the reason and the number of rows affected.
- **Test data is never used to fit anything.**

## Planned structure

1. **Setup and data loading:** Paths, packages, loading `train`, `test` and the customer table, and a quick look at each
2. **Stateless cleaning:**
   - 2.1. Merge the customer table and check dates
   - 2.2. Removing missing records
   - 2.3. Replace placeholders with missing values
   - 2.4. Repair columns with inconsistencies
   - 2.5. Sanity check in the test set
3. **Validation design:** Rolling yearly folds for choosing between options
4. **Baseline model:** A simple model on the cleaned data, with the all-positive F1 as the floor to beat
5. **Treatment comparisons:**
   - 5.1. Initial invalid values
   - 5.2. Remaining missing values
   - 5.3. Outliers
   - 5.4. Transformations and scaling
   - 5.5. Categorical encoding
   - 5.6. Customer-table field
6. **Feature engineering:** Ratios, rates and recency measures
7. **Feature selection:** Analyzing and selecting features based on performances
8. **Final preprocessing and outputs:** Cleaned files and the log.
9. **Summary of decisions.**

# 1. Setup and data loading

In [23]:
# Handle file paths without relying on computer-specific locations.
from pathlib import Path

# Locate the project from the current folder or one of its parent folders.
PROJECT_ROOT = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "data").is_dir()
        and (folder / "notebooks").is_dir()
    ),
    None,
)

# Give a clear error if the expected project structure cannot be found.
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate the project. Open the notebook within the repository."
    )

DATA_DIR = PROJECT_ROOT / "data"

# Check the expected datasets without loading their contents.
DATA_FILES = [
    "train.csv",
    "test.csv",
    "clientes.csv",
    "sample_submission.csv",
]

total_size_mib = 0

for filename in DATA_FILES:
    file_path = DATA_DIR / filename

    if not file_path.is_file():
        raise FileNotFoundError(f"Missing dataset: {file_path}")

    # Convert file size from bytes to mebibytes (MiB).
    size_mib = file_path.stat().st_size / (1024 ** 2)
    total_size_mib += size_mib

    print(f"{filename}: {size_mib:,.2f} MiB")

print(f"\nTotal size: {total_size_mib:,.2f} MiB")

train.csv: 2.39 MiB
test.csv: 0.36 MiB
clientes.csv: 0.34 MiB
sample_submission.csv: 0.01 MiB

Total size: 3.11 MiB


## 1.1. Imports and Display Settings

Import the libraries used throughout the analysis and configure display
settings to keep tables readable and plots consistent.

In [24]:
# Standard library
import sys
import warnings

# Data manipulation and numerical calculations
import pandas as pd
import numpy as np

# Charts
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical functions
from scipy import stats

# Notebook display
from IPython.display import display

# Persist fitted preprocessing steps
import joblib

# Custom stateless transformer base classes (DataRepair)
from sklearn.base import BaseEstimator, TransformerMixin, clone

# Stateful preprocessing, fitted inside each training fold only
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    RobustScaler,
    OneHotEncoder,
    TargetEncoder,
    FunctionTransformer,
)

# Reference models: linear branch and tree branch
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

# Metrics: F1 for the decision, threshold-free metrics for ranking
from sklearn.metrics import f1_score, log_loss, roc_auc_score

# Importing the log
sys.path.insert(0, str(PROJECT_ROOT))
from src.decision_log import DecisionLog

# Reproducibility
RANDOM_STATE = 42

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", "{:,.3f}".format)

# Plot style
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

# Impossible values found in EDA
PLACEHOLDER_DAYS = -4
PLACEHOLDER_AMOUNT = 1_000_000_000

## 1.2. Data Loading

In [25]:
# Load the datasets into separate DataFrames
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
clientes = pd.read_csv(DATA_DIR / "clientes.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

# Group the datasets to inspect them consistently
datasets = {
    "train": train,
    "test": test,
    "clientes": clientes,
    "sample_submission": sample_submission,
}

# Show the dimensions, column names and first five rows of each dataset
for name, df in datasets.items():
    print(f"\nDataset: {name}")
    print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]:,}")
    print(f"Columns: {df.columns.tolist()}")

    display(df.head())


Dataset: train
Rows: 6,534 | Columns: 39
Columns: ['Total a Pagar', 'Total Bruto', 'Total Liquido', 'Total Imp_', 'Total Portes', 'Total Desc_ Global', 'Total Desc_ Linha', 'Custo', 'N_ Detalhes', 'invoice_count_current_purchase', 'repeat_purchase_90d', 'days_since_previous_purchase', 'customer_tenure_days', 'prior_purchase_count', 'prior_total_spend', 'prior_mean_purchase_value', 'prior_std_purchase_value', 'prior_min_purchase_value', 'prior_max_purchase_value', 'prior_mean_gap_days', 'prior_std_gap_days', 'purchase_count_30d', 'spend_30d', 'purchase_count_90d', 'spend_90d', 'purchase_count_180d', 'spend_180d', 'purchase_count_365d', 'spend_365d', 'customer_id', 'payment_type', 'salesperson', 'commercial_zone', 'transport_method', 'warehouse', 'document_series', 'purchase_date', 'ID', 'random_noise']


,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,repeat_purchase_90d,days_since_previous_purchase,customer_tenure_days,prior_purchase_count,prior_total_spend,prior_mean_purchase_value,prior_std_purchase_value,prior_min_purchase_value,prior_max_purchase_value,prior_mean_gap_days,prior_std_gap_days,purchase_count_30d,spend_30d,purchase_count_90d,spend_90d,purchase_count_180d,spend_180d,purchase_count_365d,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,ID,random_noise
0,558.110,510.900,453.750,104.360,0.000,0.000,57.150,153.100,4.000,1.000,1,79.000,126.000,2.000,858.300,429.150,112.656,349.490,508.810,47.000,NaN,0.000,0.000,1.000,349.490,2.000,858.300,2.000,858.300,660784,Maybe Tomorrow Payment 10,VLOOKUP Master 9,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2024-10-29,10000000,-0.660
1,978.800,830.020,795.770,183.030,0.000,0.000,34.250,472.943,32.000,1.000,1,244.000,616.000,5.000,"6,722.830","1,344.566","1,683.795",84.380,"4,222.150",93.000,117.658,0.000,0.000,0.000,0.000,0.000,0.000,4.000,"2,500.680",189146,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 36,Final Excel Warehouse 1,Ctrl Z Series 1,2017-07-08,10000001,0.925
2,10.330,12.000,8.400,1.930,0.000,0.000,3.600,6.097,1.000,1.000,1,9.000,610.000,90.000,"17,590.080",195.445,190.390,4.540,"1,002.150",6.753,6.384,2.000,111.270,9.000,"1,928.550",28.000,"5,290.250",49.000,"7,899.800",103940,Maybe Tomorrow Payment 9,VLOOKUP Master 1,Decorative Dashboard Zone 4,NaN,Final Excel Warehouse 1,Ctrl Z Series 3,2019-03-19,10000002,-0.259
3,856.830,696.610,696.610,160.220,0.000,0.000,0.000,408.167,9.000,1.000,0,97.000,97.000,1.000,270.330,270.330,NaN,270.330,270.330,NaN,NaN,0.000,0.000,0.000,0.000,1.000,270.330,1.000,270.330,512927,Maybe Tomorrow Payment 7,VLOOKUP Master 4,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 71,Final Excel Warehouse 1,Ctrl Z Series 3,2018-11-10,10000003,1.448
4,"1,616.100","1,488.850","1,313.900",302.200,0.000,0.000,174.950,485.402,33.000,1.000,1,85.000,"1,575.000",25.000,"12,803.020",512.121,406.697,22.580,"1,736.740",62.083,66.247,0.000,0.000,2.000,184.130,4.000,"2,062.500",4.000,"2,062.500",122356,Maybe Tomorrow Payment 12,NaN,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2021-11-30,10000004,-1.059



Dataset: test
Rows: 986 | Columns: 38
Columns: ['Total a Pagar', 'Total Bruto', 'Total Liquido', 'Total Imp_', 'Total Portes', 'Total Desc_ Global', 'Total Desc_ Linha', 'Custo', 'N_ Detalhes', 'invoice_count_current_purchase', 'days_since_previous_purchase', 'customer_tenure_days', 'prior_purchase_count', 'prior_total_spend', 'prior_mean_purchase_value', 'prior_std_purchase_value', 'prior_min_purchase_value', 'prior_max_purchase_value', 'prior_mean_gap_days', 'prior_std_gap_days', 'purchase_count_30d', 'spend_30d', 'purchase_count_90d', 'spend_90d', 'purchase_count_180d', 'spend_180d', 'purchase_count_365d', 'spend_365d', 'customer_id', 'payment_type', 'salesperson', 'commercial_zone', 'transport_method', 'warehouse', 'document_series', 'purchase_date', 'ID', 'random_noise']


,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,days_since_previous_purchase,customer_tenure_days,prior_purchase_count,prior_total_spend,prior_mean_purchase_value,prior_std_purchase_value,prior_min_purchase_value,prior_max_purchase_value,prior_mean_gap_days,prior_std_gap_days,purchase_count_30d,spend_30d,purchase_count_90d,spend_90d,purchase_count_180d,spend_180d,purchase_count_365d,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,ID,random_noise
0,286.370,582.050,232.820,53.550,0.000,0.000,349.230,104.409,22,1,40.000,3905,122,"46,287.230",379.404,528.298,2.720,"3,198.000",31.942,48.035,0,0.000,6,"1,295.820",9,"2,498.900",23,"9,140.120",519184,Maybe Tomorrow Payment 12,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 80,Final Excel Warehouse 1,Ctrl Z Series 2,2026-08-22,10000006,0.622
1,"1,949.250","1,725.000","1,725.000",224.250,0.000,0.000,0.000,0.000,1,1,63.000,3613,58,"243,340.480","4,195.526","2,545.945",569.520,"11,031.740",62.281,46.416,0,0.000,1,"3,137.110",2,"5,086.360",4,"16,252.350",942149,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,NaN,Final Excel Warehouse 1,Ctrl Z Series 2,2025-07-26,10000008,-1.319
2,"2,738.710","4,114.830","2,738.710",0.000,0.000,0.000,"1,376.120","1,019.467",99,1,230.000,1786,16,"21,570.230","1,348.139","1,096.651",40.090,"3,363.230",103.733,78.583,0,0.000,0,0.000,0,0.000,4,"4,570.760",386654,Maybe Tomorrow Payment 4,VLOOKUP Master 1,Decorative Dashboard Zone 2,Maybe Tomorrow Delivery 54,Final Excel Warehouse 1,Ctrl Z Series 2,2026-08-22,10000010,1.252
3,726.980,591.040,591.040,135.940,0.000,0.000,0.000,341.649,39,2,234.000,374,3,"3,907.610","1,302.537",881.237,296.370,"1,937.150",70.000,67.882,0,0.000,0,0.000,0,0.000,2,"1,970.460",588088,Maybe Tomorrow Payment 7,VLOOKUP Master 4,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 35,Final Excel Warehouse 1,Ctrl Z Series 2,2025-11-05,10000020,0.988
4,670.760,555.490,545.330,125.430,0.000,0.000,10.160,326.716,13,1,NaN,386,5,"4,877.080",975.416,334.407,546.280,"1,305.090",77.000,43.825,0,0.000,1,759.060,2,"2,062.370",4,"4,330.800",610152,Maybe Tomorrow Payment 7,VLOOKUP Master 4,Decorative Dashboard Zone 1,Maybe Tomorrow Delivery 73,Final Excel Warehouse 1,Ctrl Z Series 2,2026-07-12,10000025,0.512



Dataset: clientes
Rows: 1,143 | Columns: 13
Columns: ['customer_id', 'country_or_region', 'commercial_region', 'island', 'municipality', 'city', 'postal_area', 'postal_code', 'street', 'address', 'customer_type', 'company_name', 'customer_since_year']


,customer_id,country_or_region,commercial_region,island,municipality,city,postal_area,postal_code,street,address,customer_type,company_name,customer_since_year
0,842570,Country CT11: Land of The Final Version,Region RG37: We Will See Later,NaN,NaN,City CY417: Missing Cell Port,NaN,Postal code PC828: #REF! Delivery Route,NaN,Address AD1120: Cell Without a Home,NaN,Excel Is Not A Database Ltd 150,2018
1,770941,Country CT9: Land of The Final Version,NaN,NaN,NaN,NaN,NaN,Postal code PC56: #REF! Delivery Route,NaN,Address AD62: Cell Without a Home,NaN,Excel Is Not A Database Ltd 1064,2021
2,827053,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC827: #REF! Delivery Route,Street ST170: PowerPoint New Walk,Address AD449: Cell Without a Home,NaN,Excel Is Not A Database Ltd 113,2016
3,494648,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC826: #REF! Delivery Route,NaN,Address AD454: Cell Without a Home,NaN,Excel Is Not A Database Ltd 430,2015
4,429291,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC826: #REF! Delivery Route,NaN,Address AD453: Cell Without a Home,NaN,Excel Is Not A Database Ltd 673,2015



Dataset: sample_submission
Rows: 986 | Columns: 2
Columns: ['ID', 'repeat_purchase_90d']


,ID,repeat_purchase_90d
0,10000006,1
1,10000008,1
2,10000010,1
3,10000020,0
4,10000025,1


# 2. Stateless cleaning
## 2.1. Merge the customer table and check dates

The customer table (`clientes`) adds one row of attributes per customer. It is joined to `train` and `test` with a left join on `customer_id`.

Dates are parsed with a fixed format so that a malformed value fails instead of becoming a silent missing value. Rows are then sorted by `purchase_date`, with `ID` as a tie-breaker, because it is time-dependent.

This step also starts the decision log. Every change to the data is recorded with its reason and the number of rows affected in train and test.

In [26]:
# Call the log helper
log = DecisionLog()

# Creating copies from the raw file
train_clean = train.copy()
test_clean = test.copy()

# The customer table must have one row per customer, or the merge duplicates rows
assert clientes["customer_id"].is_unique, "clientes has repeated customer_id"

# Only the key may be shared between the tables
shared = (set(train.columns) | set(test.columns)) & set(clientes.columns)
assert shared == {"customer_id"}, f"Unexpected shared columns: {shared}"

# Coverage before merging: how many rows have a customer record
matched = {}
for name, df in [("train", train_clean), ("test", test_clean)]:
    matched[name] = int(df["customer_id"].isin(clientes["customer_id"]).sum())
    print(f"{name}: {matched[name]:,} of {len(df):,} rows have a customer record")

train: 6,534 of 6,534 rows have a customer record
test: 986 of 986 rows have a customer record


In [27]:
# Left join keeps every purchase row & many_to_one ensures no duplicates.
n_train, n_test = len(train_clean), len(test_clean)
train_clean = train_clean.merge(clientes, on="customer_id", how="left", validate="many_to_one")
test_clean = test_clean.merge(clientes, on="customer_id", how="left", validate="many_to_one")
assert len(train_clean) == n_train and len(test_clean) == n_test, "Merge changed row count"

log.record(
    "D1", "customer_id", "left join clientes (many_to_one)",
    "Add customer attributes without changing the number of purchase rows.",
    matched["train"], matched["test"],
    carries={"on": "customer_id", "how": "left", "validate": "many_to_one"},
)

In [28]:
# Strict date parsing: a wrong format raises an error, missing stays NaN.
for df in (train_clean, test_clean):
    df["purchase_date"] = pd.to_datetime(df["purchase_date"], format="%Y-%m-%d")

log.record(
    "D2", "purchase_date", "parse as datetime, format %Y-%m-%d",
    "Time order is needed for validation; strict format avoids silent errors.",
    int(train_clean["purchase_date"].notna().sum()),
    int(test_clean["purchase_date"].notna().sum()),
    carries={"format": "%Y-%m-%d"},
)

# Date summary per dataset.
date_summary = pd.DataFrame(
    {
        name: {
            "rows": len(df),
            "missing_date": int(df["purchase_date"].isna().sum()),
            "first_date": df["purchase_date"].min().date(),
            "last_date": df["purchase_date"].max().date(),
        }
        for name, df in [("train", train_clean), ("test", test_clean)]
    }
).T
display(date_summary)

,rows,missing_date,first_date,last_date
train,6534,65,2015-07-11,2025-02-15
test,986,0,2025-05-24,2026-10-06


In [29]:
# Test should come after train in time
train_last = train_clean["purchase_date"].max()
test_first = test_clean["purchase_date"].min()
print(f"Train last date: {train_last.date()} | Test first date: {test_first.date()}")
print(f"Test starts after train ends: {test_first > train_last}")

# Customer cannot buy before the year they became a customer
for name, df in [("train", train_clean), ("test", test_clean)]:
    too_early = int((df["purchase_date"].dt.year < df["customer_since_year"]).sum())
    print(f"{name}: {too_early:,} rows with purchase year before customer_since_year")

# ID must be unique across both files 
all_ids = pd.concat([train_clean["ID"], test_clean["ID"]])
assert all_ids.is_unique, "Repeated ID across train/test"

# Sort by time. Missing dates go last
train_clean = train_clean.sort_values(["purchase_date", "ID"], kind="stable").reset_index(drop=True)
test_clean = test_clean.sort_values(["purchase_date", "ID"], kind="stable").reset_index(drop=True)

log.record(
    "D3", "purchase_date", "sort by purchase_date then ID",
    "Time-ordered folds need rows in date order; ID makes ties deterministic.",
    len(train_clean), len(test_clean),
    carries={"by": ["purchase_date", "ID"]},
)

display(log.to_frame())

Train last date: 2025-02-15 | Test first date: 2025-05-24
Test starts after train ends: True
train: 0 rows with purchase year before customer_since_year
test: 0 rows with purchase year before customer_since_year


,step,column,action,reason,rows_train,rows_test
0,D1,customer_id,left join clientes (many_to_one),Add customer attributes without changing the number of purchase rows.,6534,986
1,D2,purchase_date,"parse as datetime, format %Y-%m-%d",Time order is needed for validation; strict format avoids silent errors.,6469,986
2,D3,purchase_date,sort by purchase_date then ID,Time-ordered folds need rows in date order; ID makes ties deterministic.,6534,986


## 2.2. Removing missing records

65 rows in `train` have no `purchase_date`. Without a date a row cannot be placed in time, so it cannot be used for validation. Before dropping them, let's look at what else is filled in on those rows.

In [30]:
# Rows with no purchase date
no_date = train_clean["purchase_date"].isna()
print(f"Rows without a date: {no_date.sum()}")

# Original columns only, leaving out keys, target and date
purchase_cols = [
    c for c in train.columns
    if c not in ["ID", "customer_id", "repeat_purchase_90d", "purchase_date"]
]

# Which columns still have a value on these rows?
filled = train_clean.loc[no_date, purchase_cols].notna().sum()
print("\nColumns with a value on these rows:")
print(filled[filled > 0])

# What does the target look like on these rows?
print("\nTarget on these rows:")
print(train_clean.loc[no_date, "repeat_purchase_90d"].value_counts().to_dict())

Rows without a date: 65

Columns with a value on these rows:
Series([], dtype: int64)

Target on these rows:
{1: 33, 0: 32}


Rows with no date are removed from `train`. `test` has no missing dates, so it stays as it is.

In [31]:
# Drop the rows without a date
n_before = len(train_clean)
train_clean = train_clean.loc[~no_date].reset_index(drop=True)
n_dropped = n_before - len(train_clean)
print(f"Rows removed: {n_dropped} | Rows left: {len(train_clean):,}")

# Both files should now have a date on every row
assert train_clean["purchase_date"].notna().all()
assert test_clean["purchase_date"].notna().all()

# Test has no missing dates, so nothing is dropped there
log.record(
    "D4", "purchase_date", "drop rows with no purchase_date",
    "Rows without a date cannot be placed in time. Train only, test has none.",
    n_dropped, 0,
    carries={"drop_if_missing": "purchase_date"},
)

display(log.to_frame())

Rows removed: 65 | Rows left: 6,469


,step,column,action,reason,rows_train,rows_test
0,D1,customer_id,left join clientes (many_to_one),Add customer attributes without changing the number of purchase rows.,6534,986
1,D2,purchase_date,"parse as datetime, format %Y-%m-%d",Time order is needed for validation; strict format avoids silent errors.,6469,986
2,D3,purchase_date,sort by purchase_date then ID,Time-ordered folds need rows in date order; ID makes ties deterministic.,6534,986
3,D4,purchase_date,drop rows with no purchase_date,"Rows without a date cannot be placed in time. Train only, test has none.",65,0


## 2.3. Replacing placeholders with missing values

Two filler values showed up in the EDA. A value of -4 appears in the day-interval columns, and a value of 1,000,000,000 appears in some amount columns. A gap of -4 days or a purchase of one billion is not real, so both are turned into missing values.

The rule looks at one row at a time, so it is applied to train and test in the same way. First we count both values in every numeric column, to make sure they only show up where we expect.

In [32]:
numeric_cols = train_clean.select_dtypes("number").columns

# Test has no target column, so reindex to compare the same columns
test_numeric = test_clean.reindex(columns=numeric_cols)

# Count both filler values in every numeric column
scan = pd.DataFrame({
    "days_train": (train_clean[numeric_cols] == PLACEHOLDER_DAYS).sum(),
    "days_test": (test_numeric == PLACEHOLDER_DAYS).sum(),
    "amount_train": (train_clean[numeric_cols] == PLACEHOLDER_AMOUNT).sum(),
    "amount_test": (test_numeric == PLACEHOLDER_AMOUNT).sum(),
})

# Show only the columns where something was found
display(scan[(scan > 0).any(axis=1)])

,days_train,days_test,amount_train,amount_test
days_since_previous_purchase,16,2,0,0
prior_mean_purchase_value,0,0,16,2
prior_mean_gap_days,16,2,0,0
spend_365d,0,0,15,2


The -4 is replaced only in the two interval columns, where a negative number of days makes no sense. The billion is replaced in every column where it was found, since no real purchase gets near that value.

In [33]:
interval_cols = ["days_since_previous_purchase", "prior_mean_gap_days"]

# Amount columns are taken from the scan above
amount_cols = scan.index[(scan["amount_train"] + scan["amount_test"]) > 0].tolist()

jobs = [
    ("D5", interval_cols, PLACEHOLDER_DAYS, "Negative day count is impossible"),
    ("D6", amount_cols, PLACEHOLDER_AMOUNT, "A value of one billion is a filler, not a purchase"),
]

for step, cols, value, reason in jobs:
    for col in cols:
        n_train = int((train_clean[col] == value).sum())
        n_test = int((test_clean[col] == value).sum())

        train_clean[col] = train_clean[col].replace(value, np.nan)
        test_clean[col] = test_clean[col].replace(value, np.nan)

        log.record(
            step, col, f"replace {value:,} with missing", reason,
            n_train, n_test,
            carries={"column": col, "replace": value, "with": None},
        )

# Nothing should be left in the columns we cleaned
for col in interval_cols:
    assert not (train_clean[col] == PLACEHOLDER_DAYS).any()
    assert not (test_clean[col] == PLACEHOLDER_DAYS).any()
for col in amount_cols:
    assert not (train_clean[col] == PLACEHOLDER_AMOUNT).any()
    assert not (test_clean[col] == PLACEHOLDER_AMOUNT).any()

display(log.to_frame())

,step,column,action,reason,rows_train,rows_test
0,D1,customer_id,left join clientes (many_to_one),Add customer attributes without changing the number of purchase rows.,6534,986
1,D2,purchase_date,"parse as datetime, format %Y-%m-%d",Time order is needed for validation; strict format avoids silent errors.,6469,986
2,D3,purchase_date,sort by purchase_date then ID,Time-ordered folds need rows in date order; ID makes ties deterministic.,6534,986
3,D4,purchase_date,drop rows with no purchase_date,"Rows without a date cannot be placed in time. Train only, test has none.",65,0
4,D5,days_since_previous_purchase,replace -4 with missing,Negative day count is impossible,16,2
5,D5,prior_mean_gap_days,replace -4 with missing,Negative day count is impossible,16,2
6,D6,prior_mean_purchase_value,"replace 1,000,000,000 with missing","A value of one billion is a filler, not a purchase",16,2
7,D6,spend_365d,"replace 1,000,000,000 with missing","A value of one billion is a filler, not a purchase",15,2


## 2.4. Repairing inconsistent columns

Several columns can be checked against each other. Past total spend, past purchase count and past mean purchase should agree. The 365-day spend should sit between the 180-day spend and the lifetime spend. The gaps between purchases should add up to the customer's history.

When two columns disagree, one of them is wrong. Before repairing anything, each rule is tested on the data to see how often it holds. A rule that holds in almost every row is safe to use for repair. A rule that holds half the time is not.

In [34]:
# Mean purchase value and single-purchase rows
TOL = 0.01  # tolerance for cent rounding

for name, df in [("train", train_clean), ("test", test_clean)]:
    count = df["prior_purchase_count"]
    mean = df["prior_mean_purchase_value"]
    expected = df["prior_total_spend"] / count

    has_prior = count > 0
    comparable = has_prior & mean.notna() & expected.notna()
    matches = comparable & ((mean - expected).abs() <= TOL)

    print(f"--- {name} ---")
    print(f"Rows with a prior purchase:        {has_prior.sum():,}")
    print(f"Mean and total both usable:        {comparable.sum():,}")
    print(f"  mean = total / count:            {matches.sum():,}")
    print(f"  mean disagrees:                  {(comparable & ~matches).sum():,}")
    print(f"  of those, negative mean:         {(comparable & ~matches & (mean < 0)).sum():,}")
    print(f"Mean missing, total available:     {(has_prior & mean.isna() & expected.notna()).sum():,}")
    print(f"Mean missing, total missing:       {(has_prior & mean.isna() & expected.isna()).sum():,}")

    # Rows with exactly one prior purchase: total, mean, min, max should be equal
    one = count == 1
    cols = ["prior_total_spend", "prior_mean_purchase_value",
            "prior_min_purchase_value", "prior_max_purchase_value"]
    usable = one & df[cols].notna().all(axis=1)
    spread = df.loc[usable, cols].max(axis=1) - df.loc[usable, cols].min(axis=1)
    print(f"Single-purchase rows:              {one.sum():,}")
    print(f"  with all four values present:    {usable.sum():,}")
    print(f"  all four equal:                  {(spread <= TOL).sum():,}")
    print()

--- train ---
Rows with a prior purchase:        5,866
Mean and total both usable:        5,656
  mean = total / count:            5,594
  mean disagrees:                  62
  of those, negative mean:         16
Mean missing, total available:     210
Mean missing, total missing:       0
Single-purchase rows:              445
  with all four values present:    424
  all four equal:                  417

--- test ---
Rows with a prior purchase:        964
Mean and total both usable:        932
  mean = total / count:            923
  mean disagrees:                  9
  of those, negative mean:         2
Mean missing, total available:     32
Mean missing, total missing:       0
Single-purchase rows:              24
  with all four values present:    23
  all four equal:                  23



In [35]:
# Maximum and 365-day spend
for name, df in [("train", train_clean), ("test", test_clean)]:
    print(f"--- {name} ---")

    # The largest past purchase cannot exceed the total of all past purchases
    too_big = df["prior_max_purchase_value"] > df["prior_total_spend"] + TOL
    print(f"prior_max above prior_total_spend: {too_big.sum():,}")

    # Four ways spend_365d can be wrong, plus the missing values from the placeholder step
    s365, s180 = df["spend_365d"], df["spend_180d"]
    c365, c180 = df["purchase_count_365d"], df["purchase_count_180d"]
    conditions = {
        "missing": s365.isna(),
        "negative": s365 < 0,
        "below spend_180d": s365 < s180 - TOL,
        "above prior_total_spend": s365 > df["prior_total_spend"] + TOL,
        "positive with 0 purchases": (s365 > 0) & (c365 == 0),
    }
    for label, mask in conditions.items():
        print(f"spend_365d {label:<26}{mask.sum():,}")

    invalid = pd.concat(conditions, axis=1).any(axis=1)

    # Count identities that let us rebuild the value
    rebuildable = invalid & (
        (c365 == c180) | (c365 == df["prior_purchase_count"]) | (c365 == 0)
    )
    print(f"spend_365d invalid rows:           {invalid.sum():,}")
    print(f"  rebuildable from counts:         {rebuildable.sum():,}")
    print()

--- train ---
prior_max above prior_total_spend: 45
spend_365d missing                   210
spend_365d negative                  15
spend_365d below spend_180d          15
spend_365d above prior_total_spend   39
spend_365d positive with 0 purchases 0
spend_365d invalid rows:           264
  rebuildable from counts:         108

--- test ---
prior_max above prior_total_spend: 6
spend_365d missing                   32
spend_365d negative                  2
spend_365d below spend_180d          2
spend_365d above prior_total_spend   7
spend_365d positive with 0 purchases 0
spend_365d invalid rows:           41
  rebuildable from counts:         5



In [36]:
# Gap identity
for name, df in [("train", train_clean), ("test", test_clean)]:
    count = df["prior_purchase_count"]
    gap_sum = df["prior_mean_gap_days"] * (count - 1)
    history = df["customer_tenure_days"] - df["days_since_previous_purchase"]

    # Needs at least two past purchases to have a gap
    comparable = (count >= 2) & gap_sum.notna() & history.notna()
    matches = comparable & ((gap_sum - history).abs() <= 0.5)

    print(f"--- {name} ---")
    print(f"Rows with two or more prior purchases: {(count >= 2).sum():,}")
    print(f"  identity can be checked:             {comparable.sum():,}")
    print(f"  identity holds:                      {matches.sum():,}")
    print(f"  identity fails:                      {(comparable & ~matches).sum():,}")
    print(f"Mean gap missing, rebuildable:         "
          f"{((count >= 2) & df['prior_mean_gap_days'].isna() & history.notna()).sum():,}")
    print()

--- train ---
Rows with two or more prior purchases: 5,421
  identity can be checked:             5,031
  identity holds:                      5,031
  identity fails:                      0
Mean gap missing, rebuildable:         203

--- test ---
Rows with two or more prior purchases: 940
  identity can be checked:             877
  identity holds:                      877
  identity fails:                      0
Mean gap missing, rebuildable:         32



In [37]:
for name, df in [("train", train_clean), ("test", test_clean)]:
    count = df["prior_purchase_count"]
    mean = df["prior_mean_purchase_value"]
    lo, hi = df["prior_min_purchase_value"], df["prior_max_purchase_value"]
    expected = df["prior_total_spend"] / count

    # Rows where recorded mean and total/count disagree
    bad = (count > 0) & mean.notna() & expected.notna() & ((mean - expected).abs() > TOL)
    has_range = bad & lo.notna() & hi.notna()

    # A true mean has to sit between the smallest and largest purchase
    def inside(v):
        return (v >= lo - TOL) & (v <= hi + TOL)

    print(f"--- {name} ---")
    print(f"Disagreeing rows with min and max:    {has_range.sum()}")
    print(f"  only total/count fits the range:    {(has_range & inside(expected) & ~inside(mean)).sum()}")
    print(f"  only recorded mean fits:            {(has_range & ~inside(expected) & inside(mean)).sum()}")
    print(f"  both fit:                           {(has_range & inside(expected) & inside(mean)).sum()}")
    print(f"  neither fits:                       {(has_range & ~inside(expected) & ~inside(mean)).sum()}")

    # Negative spend_365d: does the positive value pass the other checks?
    s365 = df["spend_365d"]
    neg = s365 < 0
    flipped = -s365
    flip_ok = neg & (flipped >= df["spend_180d"] - TOL) & (flipped <= df["prior_total_spend"] + TOL)
    print(f"Negative spend_365d:                  {neg.sum()}")
    print(f"  passes checks after flipping sign:  {flip_ok.sum()}")

    # spend_365d above lifetime spend: is the 365-day count also above the lifetime count?
    above = s365 > df["prior_total_spend"] + TOL
    count_above = above & (df["purchase_count_365d"] > count)
    print(f"spend_365d above prior_total_spend:   {above.sum()}")
    print(f"  365-day count also above lifetime:  {count_above.sum()}")
    print()

--- train ---
Disagreeing rows with min and max:    62
  only total/count fits the range:    61
  only recorded mean fits:            0
  both fit:                           1
  neither fits:                       0
Negative spend_365d:                  15
  passes checks after flipping sign:  14
spend_365d above prior_total_spend:   39
  365-day count also above lifetime:  0

--- test ---
Disagreeing rows with min and max:    9
  only total/count fits the range:    8
  only recorded mean fits:            0
  both fit:                           1
  neither fits:                       0
Negative spend_365d:                  2
  passes checks after flipping sign:  2
spend_365d above prior_total_spend:   7
  365-day count also above lifetime:  0



The checks point to the same pattern each time. When mean and total disagree, total / count falls between the smallest and largest purchase and the recorded mean does not. So the total is right and the mean is recomputed from it. The negative 365-day spends become valid after flipping the sign, the same mistake seen in the mean. When the 365-day spend is above the lifetime spend while the counts agree, the spend value is the broken one.

Every repair adds a flag column. Whether the flags help the model is tested in section 5.

One rule is checked before it is used. With exactly one past purchase, the days since the previous purchase should equal the customer's tenure.

In [38]:
# With one past purchase, the previous purchase is also the first one
for name, df in [("train", train_clean), ("test", test_clean)]:
    one = (df["prior_purchase_count"] == 1) & df["days_since_previous_purchase"].notna()
    same = one & ((df["days_since_previous_purchase"] - df["customer_tenure_days"]).abs() <= 0.5)
    print(f"{name}: {same.sum()} of {one.sum()} single-purchase rows have days_since = tenure")


train: 423 of 423 single-purchase rows have days_since = tenure
test: 23 of 23 single-purchase rows have days_since = tenure


In [39]:
def repair_frame(df, use_single_days_rule):
    """Row-by-row repairs. Returns the repaired copy and the rows changed per step."""
    df = df.copy()
    n = {}
    count = df["prior_purchase_count"]
    total = df["prior_total_spend"]

    # Mean purchase value: recompute from total and count
    old_mean = df["prior_mean_purchase_value"]
    new_mean = total / count
    fix_mean = (count > 0) & new_mean.notna() & (
        old_mean.isna() | ((old_mean - new_mean).abs() > TOL)
    )
    df.loc[fix_mean, "prior_mean_purchase_value"] = new_mean[fix_mean]
    n["mean"] = int(fix_mean.sum())

    # One past purchase: mean, min and max all equal the total
    single_cols = ["prior_mean_purchase_value", "prior_min_purchase_value", "prior_max_purchase_value"]
    one = (count == 1) & total.notna()
    off = df[single_cols].sub(total, axis=0).abs().gt(TOL) | df[single_cols].isna()
    fix_single = one & off.any(axis=1)
    for col in single_cols:
        df.loc[one, col] = total[one]
    n["single"] = int(fix_single.sum())

    # Largest purchase above the total is impossible
    bad_max = df["prior_max_purchase_value"] > total + TOL
    df.loc[bad_max, "prior_max_purchase_value"] = np.nan
    n["max"] = int(bad_max.sum())

    # 365-day spend: flip negatives that pass the checks once positive
    s180 = df["spend_180d"]
    s365 = df["spend_365d"]
    flip = (s365 < 0) & (-s365 >= s180 - TOL) & (-s365 <= total + TOL)
    df.loc[flip, "spend_365d"] = -s365[flip]
    n["flip"] = int(flip.sum())

    # 365-day spend: find what is still invalid and rebuild from the counts
    s365 = df["spend_365d"]
    c365, c180 = df["purchase_count_365d"], df["purchase_count_180d"]
    invalid = (
        s365.isna() | (s365 < 0) | (s365 < s180 - TOL)
        | (s365 > total + TOL) | ((s365 > 0) & (c365 == 0))
    )
    rebuilt = pd.Series(
        np.select([c365 == 0, c365 == c180, c365 == count], [0.0, s180, total], default=np.nan),
        index=df.index,
    )
    got_value = invalid & rebuilt.notna()
    blanked = invalid & s365.notna() & rebuilt.isna()
    df.loc[invalid, "spend_365d"] = rebuilt[invalid]
    n["s365_rebuilt"] = int(got_value.sum())
    n["s365_blanked"] = int(blanked.sum())

    # Mean gap between purchases: rebuild from tenure and days since previous
    tenure = df["customer_tenure_days"]
    days = df["days_since_previous_purchase"]
    gap_value = (tenure - days) / (count - 1)
    fix_gap = (count >= 2) & df["prior_mean_gap_days"].isna() & gap_value.notna() & (gap_value >= 0)
    df.loc[fix_gap, "prior_mean_gap_days"] = gap_value[fix_gap]
    n["gap"] = int(fix_gap.sum())

    # Days since previous purchase: same identity, solved the other way
    gap = df["prior_mean_gap_days"]
    days_value = tenure - gap * (count - 1)
    fix_days = (count >= 2) & days.isna() & days_value.notna() & (days_value >= 0)
    if use_single_days_rule:
        single_days = (count == 1) & days.isna() & tenure.notna()
        days_value = days_value.where(~single_days, tenure)
        fix_days = fix_days | single_days
    df.loc[fix_days, "days_since_previous_purchase"] = days_value[fix_days]
    n["days"] = int(fix_days.sum())

    # One flag per repair, plus one for any repair
    flags = {
        "flag_mean_repaired": fix_mean | fix_single,
        "flag_max_invalid": bad_max,
        "flag_spend365_repaired": flip | invalid,
        "flag_gap_rebuilt": fix_gap | fix_days,
    }
    for col, mask in flags.items():
        df[col] = mask.astype(int)
    df["flag_had_repair"] = pd.concat(flags, axis=1).any(axis=1).astype(int)

    return df, n

The same function runs on train and test, and each step goes into the log with the number of rows it changed.

In [40]:
# True only if almost every row matched
USE_SINGLE_DAYS_RULE = True

train_clean, n_tr = repair_frame(train_clean, USE_SINGLE_DAYS_RULE)
test_clean, n_te = repair_frame(test_clean, USE_SINGLE_DAYS_RULE)

steps = [
    ("D7", "prior_mean_purchase_value", "mean", "recompute as total / count",
     "Total / count fits the min-max range; the recorded mean often does not."),
    ("D8", "prior_mean/min/max_purchase_value", "single", "set equal to total when count is 1",
     "With one purchase, mean, min and max are that purchase."),
    ("D9", "prior_max_purchase_value", "max", "set to missing when above total",
     "The largest purchase cannot exceed the sum of all purchases."),
    ("D10", "spend_365d", "flip", "flip sign of negatives that pass checks",
     "Same sign error as the mean; the positive value fits the 180-day and lifetime bounds."),
    ("D11", "spend_365d", "s365_rebuilt", "rebuild invalid values from count identities",
     "Equal counts mean equal spend: 0, spend_180d or prior_total_spend."),
    ("D12", "spend_365d", "s365_blanked", "set to missing when invalid and not rebuildable",
     "No rule gives a reliable value; imputation happens inside the folds."),
    ("D13", "prior_mean_gap_days", "gap", "rebuild from (tenure - days_since) / (count - 1)",
     "The identity held in every checkable row of train and test."),
    ("D14", "days_since_previous_purchase", "days", "rebuild from tenure - gap * (count - 1)",
     "Same identity; with one purchase, days_since equals tenure."),
]

for step, col, key, action, reason in steps:
    log.record(step, col, action, reason, n_tr[key], n_te[key], carries={"rule": key})

display(log.to_frame())

,step,column,action,reason,rows_train,rows_test
0,D1,customer_id,left join clientes (many_to_one),Add customer attributes without changing the number of purchase rows.,6534,986
1,D2,purchase_date,"parse as datetime, format %Y-%m-%d",Time order is needed for validation; strict format avoids silent errors.,6469,986
2,D3,purchase_date,sort by purchase_date then ID,Time-ordered folds need rows in date order; ID makes ties deterministic.,6534,986
3,D4,purchase_date,drop rows with no purchase_date,"Rows without a date cannot be placed in time. Train only, test has none.",65,0
4,D5,days_since_previous_purchase,replace -4 with missing,Negative day count is impossible,16,2
5,D5,prior_mean_gap_days,replace -4 with missing,Negative day count is impossible,16,2
6,D6,prior_mean_purchase_value,"replace 1,000,000,000 with missing","A value of one billion is a filler, not a purchase",16,2
7,D6,spend_365d,"replace 1,000,000,000 with missing","A value of one billion is a filler, not a purchase",15,2
8,D7,prior_mean_purchase_value,recompute as total / count,Total / count fits the min-max range; the recorded mean often does not.,272,41
9,D8,prior_mean/min/max_purchase_value,set equal to total when count is 1,"With one purchase, mean, min and max are that purchase.",3,0


In [41]:
# Rerun the checks on the repaired data: every count should be 0
for name, df in [("train", train_clean), ("test", test_clean)]:
    count = df["prior_purchase_count"]
    total = df["prior_total_spend"]
    s365 = df["spend_365d"]
    gap_sum = df["prior_mean_gap_days"] * (count - 1)
    history = df["customer_tenure_days"] - df["days_since_previous_purchase"]

    checks = {
        "mean differs from total / count": ((count > 0) & ((df["prior_mean_purchase_value"] - total / count).abs() > TOL)).sum(),
        "max above total": (df["prior_max_purchase_value"] > total + TOL).sum(),
        "spend_365d negative": (s365 < 0).sum(),
        "spend_365d below spend_180d": (s365 < df["spend_180d"] - TOL).sum(),
        "spend_365d above total": (s365 > total + TOL).sum(),
        "gap identity fails": ((count >= 2) & ((gap_sum - history).abs() > 0.5)).sum(),
    }
    print(f"--- {name} ---")
    for label, value in checks.items():
        print(f"{label:<34}{value}")

    # Missing values left for imputation
    cols = ["prior_mean_purchase_value", "prior_max_purchase_value", "spend_365d",
            "prior_mean_gap_days", "days_since_previous_purchase"]
    print("Missing after repair:", df[cols].isna().sum().to_dict())
    print()

--- train ---
mean differs from total / count   0
max above total                   0
spend_365d negative               0
spend_365d below spend_180d       0
spend_365d above total            0
gap identity fails                0
Missing after repair: {'prior_mean_purchase_value': 603, 'prior_max_purchase_value': 645, 'spend_365d': 145, 'prior_mean_gap_days': 1056, 'days_since_previous_purchase': 611}

--- test ---
mean differs from total / count   0
max above total                   0
spend_365d negative               0
spend_365d below spend_180d       0
spend_365d above total            0
gap identity fails                0
Missing after repair: {'prior_mean_purchase_value': 22, 'prior_max_purchase_value': 28, 'spend_365d': 34, 'prior_mean_gap_days': 46, 'days_since_previous_purchase': 22}



Every rule holds after the repair, in train and in test. Most of the remaining missing values are expected rather than errors. A customer with no past purchase has no mean purchase value, no largest purchase and no previous purchase date. A customer with one past purchase has no gap between purchases. Only a small group is missing for other reasons: 42 maximums removed in D9, 8 interval values, and 145 values of `spend_365d` that no rule could rebuild. How to fill each group is tested in the proper section.

## 2.5. Sanity check on the test set

The repairs ran on test with the same function, and the checks at the end of 2.4 came back clean for both files. Category levels, customer overlap and distribution shifts between train and test were already compared in `01_eda.ipynb` (section 12). Cleaning did not change them. One thing is left to check: after the merge and the new flag columns, train and test should still have the same columns and types.

In [42]:
target = "repeat_purchase_90d"
train_features = train_clean.drop(columns=target)

# Check columns, Check order
same_cols = list(train_features.columns) == list(test_clean.columns)
print(f"Same columns in the same order: {same_cols}")
print("Only in train:", sorted(set(train_features.columns) - set(test_clean.columns)))
print("Only in test:", sorted(set(test_clean.columns) - set(train_features.columns)))

# Dtype differences (float vs int after missing values is expected)
dtypes = pd.DataFrame({"train": train_features.dtypes.astype(str), "test": test_clean.dtypes.astype(str)})
diff = dtypes[dtypes["train"] != dtypes["test"]]
if diff.empty:
    print("All dtypes match.")
else:
    display(diff)

print(f"\nTrain: {train_clean.shape} | Test: {test_clean.shape}")

Same columns in the same order: True
Only in train: []
Only in test: []


,train,test
N_ Detalhes,float64,int64
invoice_count_current_purchase,float64,int64
customer_tenure_days,float64,int64
prior_purchase_count,float64,int64
purchase_count_30d,float64,int64
purchase_count_90d,float64,int64
purchase_count_180d,float64,int64
purchase_count_365d,float64,int64



Train: (6469, 56) | Test: (986, 55)


In [ ]:
# A second pass of the repair should change nothing
data_cols = [c for c in test_clean.columns if not c.startswith("flag_")]

for name, df in [("train", train_clean), ("test", test_clean)]:
    again, n_again = repair_frame(df, USE_SINGLE_DAYS_RULE)
    unchanged = again[data_cols].equals(df[data_cols])
    print(f"{name}: unchanged after second pass = {unchanged} | rows touched: {n_again}")